# plantid: organ isolation with LocateAnything-3B, then re-test shape descriptors and the CNN

Two recent experiments both failed for what looks like the same reason:

1. Concatenating a Hu-moment shape descriptor (computed via Otsu+center-mask on the whole field photo) with the CNN leaf embedding didn't help k-NN accuracy.
2. Adding a Canny edge-magnitude channel to the CNN's input (`ce_edge` in `finetune_colab.ipynb`) made things slightly *worse* across the board (classifier-head val acc 0.633/0.638 -> 0.597; every per-organ k-NN number a notch below CE/SupCon).

In both cases the shape/edge signal is computed over the **whole, cluttered field photo** - background grass, twigs, and other leaves dominate the edge map / outline, so the signal isn't actually about the leaf/bark/flower's margin shape. The qualitative k-NN failures that motivated this (e.g. *Chaerophyllum temulum* retrieved as *Aegopodium podagraria*, *Althaea officinalis* as *Daphne mezereum*) look like cases where the organ's outline genuinely differs but is buried in noise.

**This notebook's hypothesis**: use [LocateAnything-3B](https://huggingface.co/nvidia/LocateAnything-3B) (an open-vocabulary grounding VLM, good at cluttered scenes) to find a bounding box for the leaf/bark/flower in each photo, crop to it, and then:

- Recompute the classical Hu-moment shape descriptor on the *crop* - does shape become a meaningful k-NN signal once it's not drowning in background clutter?
- Retrain the CE-only `mobilenet_v3_small` on *cropped* images - does removing background clutter improve raw classifier-head/k-NN accuracy?
- Filter out images where the target organ doesn't take up enough of the frame to be useful (Step 6) - both to keep the crop-based experiments clean, and as a candidate input-validation check for a deployed app ("not enough leaf visible - try getting closer").
- Re-examine the two known failure cases qualitatively on crops.

Scoped to **CE-only** (no SupCon/edge channel) - those were already shown to be a wash, so this isolates the one new variable: cropping.

## Plan and runtime expectations

| Step | What | Runtime |
|---|---|---|
| 3 | Install + load LocateAnything-3B | ~2-5 min |
| 4 | Sanity-check grounding prompts on ~12 sample images | ~1 min |
| 5 | Run grounding over the dataset -> `organ_crops.parquet` | **slow** - see `MAX_PER_GROUP` below, checkpointed/resumable |
| 6 | Filter to images where the organ is clearly visible (`MIN_AREA_FRAC`) | <1 min |
| 7 | Visualize crops, incl. filtered-out examples and the two known failure-case species | ~1 min |
| 8 | **Go/no-go check**: Hu moments on crops vs classical baseline | ~1 min, CPU-only |
| 9-13 | Retrain CE on crops, compare to whole-image CE baseline | ~15 min |
| 14-15 | Qualitative classifier + k-NN inspection on crops | ~1 min |
| 16 | Export `organ_crops.parquet` + crop-trained model/embeddings | ~1 min |

Step 5 is the bottleneck: a 3B VLM forward pass per image. `MAX_PER_GROUP` caps how many images per (organ, species) get a grounding call - set it lower (e.g. 15-20) for a faster first pass, or `None` for the full ~10,777 images. Step 5 checkpoints to a parquet every 200 images and skips already-processed image_ids, so it's safe to stop and re-run the cell to resume.

## Whole-image CE baseline (for comparison; from `metadata.json` of the most recent `finetune_colab.ipynb` run)

| organ  | clf top1 | clf top5 | clf top10 | k-NN(`ce_emb`) top1 | top5 | top10 |
|--------|----------|----------|-----------|---------------------|------|-------|
| leaf   | 0.581 | 0.877 | 0.939 | 0.517 | 0.830 | 0.882 |
| bark   | 0.459 | 0.716 | 0.787 | 0.317 | 0.579 | 0.672 |
| flower | 0.716 | 0.921 | 0.958 | 0.679 | 0.894 | 0.926 |

And the Phase 2 classical-descriptor baseline (whole-image Hu/LBP/HSV k-NN): leaf 0.065/0.183/0.283, bark 0.098/0.257/0.383, flower 0.156/0.348/0.472.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## Step 1: upload the dataset manifest

Upload `plantnet_index.parquet` and `outlier_scores.parquet` from `data/processed/` (same files used by `finetune_colab.ipynb`).

All dataset state lives in a single `data` object (a `Catalog` dataclass built by `Catalog.load(...)`): `data.index` is the manifest, `data.species_to_idx`/`data.species_to_name`/`data.species_ids` are the label mappings, and Steps 5-9 fill in `data.crops_df`, `data.crops_lookup`, `data.crop_index`, and `data.train_df`/`data.val_df`/`data.test_df` as the pipeline progresses. `ORGANS = ('leaf', 'bark', 'flower')` is also defined here for reuse throughout.

This cell also mounts Google Drive (`drive.mount`) and creates `DRIVE_DIR = /content/drive/MyDrive/plantid` - a one-time auth popup per session. `organ_crops.parquet` (Step 5's output) is saved there so it survives Colab disconnects: see the note in Step 5.


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import json
import re
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.spatial.distance import cdist
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm


@dataclass
class Catalog:
    """Single object threaded through the notebook: dataset index + everything derived from it."""
    index: pd.DataFrame
    species_ids: list
    species_to_idx: dict
    species_to_name: dict
    crops_df: pd.DataFrame = None
    crops_lookup: dict = field(default_factory=dict)
    crop_index: pd.DataFrame = None
    train_df: pd.DataFrame = None
    val_df: pd.DataFrame = None
    test_df: pd.DataFrame = None

    @classmethod
    def load(cls, index_path, outliers_path):
        index = pd.read_parquet(index_path)
        outliers = pd.read_parquet(outliers_path)

        species_ids = sorted(index['species_id'].unique())
        species_to_idx = {sp: i for i, sp in enumerate(species_ids)}
        species_to_name = dict(zip(index['species_id'], index['species_name']))
        index['label'] = index['species_id'].map(species_to_idx)

        flagged = set(outliers.loc[outliers['is_outlier'], 'image_id'])
        is_train = index['split'] == 'train'
        index['use_for_train'] = is_train & ~index['image_id'].isin(flagged)

        return cls(index=index, species_ids=species_ids, species_to_idx=species_to_idx, species_to_name=species_to_name)


ORGANS = ('leaf', 'bark', 'flower')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/plantid')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

ROOT = Path('/content/plantid_data')
IMAGES_DIR = ROOT / 'images'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

data = Catalog.load('plantnet_index.parquet', 'outlier_scores.parquet')
print(data.index.shape, data.index['species_id'].nunique(), 'species')
print(data.index.groupby('split').size())
print('train (clean):', data.index['use_for_train'].sum(), '/ train (all):', (data.index['split'] == 'train').sum())
print('val:', (data.index['split'] == 'val').sum(), 'test:', (data.index['split'] == 'test').sum())


## Step 2: download images from PlantNet


In [ ]:
IMAGE_URL = 'https://bs.plantnet.org/image/m/{image_id}'
MAX_WORKERS = 64

_thread_local = threading.local()


def _session():
    sess = getattr(_thread_local, 'session', None)
    if sess is None:
        sess = requests.Session()
        adapter = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
        sess.mount('https://', adapter)
        _thread_local.session = sess
    return sess


def fetch(row):
    dest = IMAGES_DIR / row.species_id / row.organ / f'{row.image_id}.jpg'
    if dest.exists():
        return row.image_id, str(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    sess = _session()
    for attempt in range(3):
        try:
            resp = sess.get(IMAGE_URL.format(image_id=row.image_id), timeout=10)
            resp.raise_for_status()
            dest.write_bytes(resp.content)
            return row.image_id, str(dest)
        except requests.RequestException:
            time.sleep(0.5 * (attempt + 1))
    return row.image_id, None


def download_images(index):
    paths = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(fetch, row) for row in index.itertuples()]
        for fut in tqdm(as_completed(futures), total=len(futures)):
            image_id, path = fut.result()
            paths[image_id] = path

    index = index.copy()
    index['abs_path'] = index['image_id'].map(paths)
    n_failed = index['abs_path'].isna().sum()
    print(f'failed downloads: {n_failed} / {len(index)}')
    return index[index['abs_path'].notna()].reset_index(drop=True)


data.index = download_images(data.index)


## Step 3: install and load LocateAnything-3B

A lightweight inference-only install - just `transformers` (with `trust_remote_code=True`, which pulls the model's custom modeling code from the HF Hub) plus a few small deps. No need for the full `Eagle/Embodied` package install (that pulls in deepspeed/liger_kernel/gradio etc., which are training/serving extras we don't need here).

The dynamic processor module also imports `decord` (video reading) and `lmdb` even though we only feed it still images, so they're included below - import will fail with a `This modeling file requires the following packages...` error otherwise.

If this cell errors on import right after the `pip install` (e.g. a `tokenizers`/`transformers` version mismatch because an older version was already imported elsewhere in this runtime), use Colab's *Runtime > Restart session* and re-run from this cell - the upgraded packages will then import cleanly. Steps 1-2 don't need to be repeated since `index`/`abs_path` would be lost on restart only if you restart before they're computed; if you do restart after Steps 1-2, just re-run all cells from the top (the manifest upload and image downloads are quick to redo from cache/disk).


In [ ]:
!pip install -q -U "transformers==4.57.1" "tokenizers==0.22.0" "accelerate==1.5.2" timm einops einops-exts sentencepiece decord lmdb

import re
import torch
from transformers import AutoModel, AutoTokenizer, AutoProcessor

LOCATE_MODEL = 'nvidia/LocateAnything-3B'
locate_tokenizer = AutoTokenizer.from_pretrained(LOCATE_MODEL, trust_remote_code=True)
locate_processor = AutoProcessor.from_pretrained(LOCATE_MODEL, trust_remote_code=True)
locate_model = AutoModel.from_pretrained(
    LOCATE_MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True,
).to('cuda').eval()


@torch.no_grad()
def locate_query(image, prompt, max_new_tokens=128):
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': prompt},
    ]}]
    text = locate_processor.py_apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = locate_processor.process_vision_info(messages)
    inputs = locate_processor(text=[text], images=images, videos=videos, return_tensors='pt').to('cuda')
    response = locate_model.generate(
        pixel_values=inputs['pixel_values'].to(torch.bfloat16),
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        image_grid_hws=inputs.get('image_grid_hws'),
        tokenizer=locate_tokenizer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        generation_mode='hybrid',
        do_sample=False,
        verbose=False,
    )
    return response[0] if isinstance(response, tuple) else response


def parse_boxes(answer, w, h):
    boxes = []
    for m in re.finditer(r'<box><(\d+)><(\d+)><(\d+)><(\d+)></box>', answer):
        x1, y1, x2, y2 = [int(g) for g in m.groups()]
        boxes.append((x1 / 1000 * w, y1 / 1000 * h, x2 / 1000 * w, y2 / 1000 * h))
    return boxes


def detect_organ_box(image, label):
    prompt = f'Locate a single instance that matches the following description: {label}.'
    answer = locate_query(image, prompt)
    return parse_boxes(answer, *image.size)


print('LocateAnything-3B loaded.')


## Step 4: sanity-check grounding prompts on sample images

Try a few candidate prompts per organ on a couple of sample images each, and look at the boxes drawn. Pick whichever prompt consistently boxes the actual leaf/bark/flower (not the whole plant or background) and set `ORGAN_PROMPTS` in Step 5 accordingly.

Note: in this dataset, the "bark" organ label spans both true woody tree bark *and* herbaceous green stems (per `PHASE1_FINDINGS.md`) - a tree-bark-specific prompt may fail to find anything on the herbaceous species. The candidates below include a more general stem/trunk/bark phrasing for comparison; pick whichever boxes the plant's main stem structure across both kinds of species.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

ORGAN_PROMPT_CANDIDATES = {
    'leaf': ['a single leaf', 'the leaf', 'a leaf in the foreground'],
    'bark': ['the trunk or bark of the tree', 'tree bark', 'the tree trunk',
             'the stem, trunk, or bark of the plant'],
    'flower': ['a flower', 'the flower', 'a single flower'],
}


def show_prompt_candidates(data, organ, prompts, n=5, seed=0):
    sample = data.index[data.index['organ'] == organ].sample(n, random_state=seed)
    for _, row in sample.iterrows():
        img = Image.open(row['abs_path']).convert('RGB')
        fig, axes = plt.subplots(1, len(prompts), figsize=(4 * len(prompts), 4))
        for ax, prompt in zip(axes, prompts):
            ax.imshow(img)
            for (x1, y1, x2, y2) in detect_organ_box(img, prompt):
                ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                                fill=False, edgecolor='lime', linewidth=2))
            ax.set_title(f'{organ}: "{prompt}"', fontsize=9)
            ax.axis('off')
        fig.suptitle(data.species_to_name[row['species_id']])
        plt.tight_layout()
        plt.show()


for organ, prompts in ORGAN_PROMPT_CANDIDATES.items():
    show_prompt_candidates(data, organ, prompts)


## Step 5: run grounding over the dataset -> `organ_crops.parquet`

Set `ORGAN_PROMPTS` based on what looked best in Step 4. For `leaf`/`flower`, if the primary prompt's box covers (almost) the whole image (`is_full_image_box`, `FULL_IMAGE_TOL`) - i.e. the model didn't isolate anything - `FALLBACK_PROMPTS` (`'a leaf'` / `'a flower'`) is retried for that image only. `MAX_PER_GROUP` caps how many images per (organ, species) get processed - `None` runs the full ~10,777 images (slow; budget hours). A smaller value (e.g. 15-20) gives a faster first pass with a still-reasonable train/val/test split for the CNN retrain in Step 9+. Each record also stores the source image's `w`/`h`, used by Step 6 to compute how much of the frame the detected box covers.

Checkpoints to `CROPS_PATH` (`DRIVE_DIR / 'organ_crops.parquet'`, i.e. Google Drive - see Step 1) every 200 images and skips already-processed `(image_id, organ)` pairs - safe to interrupt and re-run this cell to resume, **including in a brand-new Colab session**: re-run Steps 1-2 (fast) then this cell, and with the same `ORGAN_PROMPTS`/`MAX_PER_GROUP` it will find everything already done and skip straight to `todo` being empty. This means you can tweak Step 9+ (filter threshold, model architecture, training hyperparameters) repeatedly without ever re-running the slow grounding loop. To force a full re-run (e.g. after changing `ORGAN_PROMPTS` or `MAX_PER_GROUP`), delete or rename the file in Drive first.

All constants (`ORGAN_PROMPTS`, `MAX_PER_GROUP`, `CROPS_PATH`, etc.) live at the top of the cell, with the logic split into small functions. The `done_ids`/`todo` lookup is a vectorized `MultiIndex.isin` instead of a row-wise `.apply`.

**Why more `PREFETCH_WORKERS` didn't help, and why batching made it *worse***: threads only overlap disk/JPEG decode with the GPU step - they can't speed up the GPU step itself, since it's a single model instance on a single GPU. A batched `generate()` call (tried in an earlier version of this cell) made things slower, not faster (4.7s/it vs 3.5s/it) - **confirmed via a standalone diagnostic**: `nvidia/LocateAnything-3B`'s custom `generate()` (`modeling_locateanything.py` line 327) has a hard `assert batch_size == 1, 'only batch size = 1 is supported now'`. Batching isn't implemented at all - every "batch" hit that assert and fell back to the same per-image calls as before, but only after paying for the failed attempt. Batching this model is a dead end short of forking/patching its modeling code.

**Measured findings** (single-image timing, A100): `max_new_tokens` does scale cost roughly linearly (8 tok=3.35s, 32 tok=5.00s, 128 tok=8.98s at 600px) but the bigger effect is input resolution - 600px=6.79s vs 448px=2.04s (~3x) vs 336px=1.35s vs 224px=1.44s, at `max_new_tokens=48`. The 600px->448px cliff strongly suggests an extra vision-tower tile kicks in above ~448px (PlantNet images are 600x600). 224px and 336px are roughly tied with 448px, so 448px is a reasonable floor without venturing into territory where small organs might become hard to localize.

**`detect_organ_box_fast` uses `MAX_NEW_TOKENS=32`** (no information loss - the box answer is ~20 tokens, well under the default 128) and an opt-in `GROUND_IMG_SIZE` (default `None` = full resolution, boxes are mapped back to the original image's pixel coordinates regardless of input size). Setting `GROUND_IMG_SIZE=448` cuts single-image grounding time by roughly 3x in testing, at the cost of giving the model less detail; `GROUND_IMG_SIZE=336` timed about the same as 448px. **If you try a smaller `GROUND_IMG_SIZE`**: let it process ~20-30 images, then check Step 7's visualizations - confirm boxes still look reasonable (a smaller input image could make thin bark/small flowers harder to localize) and that the `found`/fallback rates haven't dropped much. A single GPU running one 3B-model instance genuinely can't be parallelized further beyond I/O prefetch - 4-5s/image for production single-image use is unaffected by any of this either way.


In [ ]:
# ---- constants -------------------------------------------------------------
ORGAN_PROMPTS = {
    'leaf': 'a single leaf',
    'bark': 'the trunk of the plant',
    'flower': 'a single flower',
}
FALLBACK_PROMPTS = {  # retried for leaf/flower if the primary prompt boxes the whole image
    'leaf': 'a leaf',
    'flower': 'a flower',
}
FULL_IMAGE_TOL = 0.02  # box within this fraction of (0, 0, w, h) on each side counts as "whole image"
MAX_PER_GROUP = 20  # images per (organ, species); set to None for the full dataset
SAMPLE_SEED = 0
CROPS_PATH = DRIVE_DIR / 'organ_crops.parquet'  # persisted in Drive: re-running this cell in a
                                                # later session resumes/skips instead of redoing grounding
CROPS_COLUMNS = ['image_id', 'organ', 'x1', 'y1', 'x2', 'y2', 'found', 'w', 'h']
CHECKPOINT_EVERY = 200
PREFETCH_WORKERS = 8  # threads for image loading/decoding, overlapped with the GPU forward pass
GROUND_IMG_SIZE = None  # set to e.g. 448 to resize before grounding (~3x faster in testing, but
                        # gives the model less detail - None keeps full resolution by default)
MAX_NEW_TOKENS = 32  # generate() token budget - the box answer is ~20 tokens, well under the default 128


# ---- functions --------------------------------------------------------------
def sample_per_group(df, max_per_group, seed=SAMPLE_SEED):
    """Cap rows per (organ, species_id) group; `None` returns df unchanged."""
    if max_per_group is None:
        return df
    return (
        df.groupby(['organ', 'species_id'], group_keys=False)
        .apply(lambda g: g.sample(min(len(g), max_per_group), random_state=seed))
    )


def load_crops_checkpoint(path, columns):
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame(columns=columns)


def filter_todo(target, crops_df):
    """Vectorized: rows of `target` whose (image_id, organ) isn't in crops_df yet."""
    target_keys = pd.MultiIndex.from_frame(target[['image_id', 'organ']])
    done_keys = pd.MultiIndex.from_frame(crops_df[['image_id', 'organ']])
    return target[~target_keys.isin(done_keys)]


def load_image(row):
    """I/O-bound: read + decode one image. Run in a thread pool to overlap with the GPU."""
    img = Image.open(row.abs_path).convert('RGB')
    return row.image_id, row.organ, img


def is_full_image_box(x1, y1, x2, y2, w, h, tol=FULL_IMAGE_TOL):
    """True if (x1,y1,x2,y2) covers ~all of (0,0,w,h) - i.e. the model didn't isolate anything."""
    return x1 <= w * tol and y1 <= h * tol and x2 >= w * (1 - tol) and y2 >= h * (1 - tol)


def detect_organ_box_fast(image, label, max_new_tokens=MAX_NEW_TOKENS, img_size=GROUND_IMG_SIZE):
    """Like Step 3's detect_organ_box, but resizes the input (the dominant cost - see Step 5 notes)
    and caps generate() length. The model's <box> output is normalized to 0-1000 regardless of
    input size, so boxes are mapped back to the ORIGINAL image's pixel space via parse_boxes(w, h)."""
    prompt = f'Locate a single instance that matches the following description: {label}.'
    w, h = image.size
    resized = image.resize((img_size, img_size)) if img_size else image
    answer = locate_query(resized, prompt, max_new_tokens=max_new_tokens)
    return parse_boxes(answer, w, h)


def ground_one(image_id, organ, img):
    """GPU-bound: a single LocateAnything-3B forward pass, with a broader fallback prompt
    for leaf/flower if the primary prompt boxes the whole image."""
    w, h = img.size
    try:
        boxes = detect_organ_box_fast(img, ORGAN_PROMPTS[organ])
    except Exception:
        boxes = []
    if boxes:
        x1, y1, x2, y2 = boxes[0]
        found = True
    else:
        x1, y1, x2, y2, found = 0, 0, w, h, False

    if found and organ in FALLBACK_PROMPTS and is_full_image_box(x1, y1, x2, y2, w, h):
        try:
            fallback_boxes = detect_organ_box_fast(img, FALLBACK_PROMPTS[organ])
        except Exception:
            fallback_boxes = []
        if fallback_boxes:
            x1, y1, x2, y2 = fallback_boxes[0]

    return {'image_id': image_id, 'organ': organ, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
            'found': found, 'w': w, 'h': h}


def save_checkpoint(crops_df, records, path):
    crops_df = pd.concat([crops_df, pd.DataFrame(records)], ignore_index=True)
    crops_df.to_parquet(path)
    return crops_df


def run_grounding(todo, crops_df, path):
    """Thread pool prefetches/decodes images while the GPU runs ground_one sequentially."""
    records = []
    rows = list(todo.itertuples(index=False))
    with ThreadPoolExecutor(max_workers=PREFETCH_WORKERS) as ex:
        for image_id, organ, img in tqdm(ex.map(load_image, rows), total=len(rows)):
            records.append(ground_one(image_id, organ, img))
            if len(records) >= CHECKPOINT_EVERY:
                crops_df = save_checkpoint(crops_df, records, path)
                records = []
    if records:
        crops_df = save_checkpoint(crops_df, records, path)
    return crops_df


# ---- run ----------------------------------------------------------------------
target = sample_per_group(data.index, MAX_PER_GROUP)
data.crops_df = load_crops_checkpoint(CROPS_PATH, CROPS_COLUMNS)
todo = filter_todo(target, data.crops_df)
print(f'{len(todo)} / {len(target)} images remaining')

data.crops_df = run_grounding(todo, data.crops_df, CROPS_PATH)
print(data.crops_df.groupby('organ')['found'].agg(['mean', 'count']))


## Step 6: filter to images where the target organ is clearly visible

LocateAnything found *some* box for most images, but a tiny box in the corner of a cluttered photo isn't useful for classification - the same way a real plantid app shouldn't try to identify a species from a photo where the leaf/bark/flower is barely in frame. For each detected box, compute its area as a fraction of the full image (`area_frac`), and require `area_frac >= MIN_AREA_FRAC` for the crop to count as "good".

This filter is also a candidate piece of UX for Phase 6 (the CLI/app): rather than silently returning a low-confidence guess for a photo where the target organ takes up only a few percent of the frame, the app could run this same check at inference time and respond with something like *"not enough leaf visible in this photo - try getting closer to the leaf"*.

Look at the per-organ histograms and pass-rates printed below and adjust `MIN_AREA_FRAC` if it's filtering too much (or too little) of the dataset.


In [ ]:
crops_df = data.crops_df
crops_df['area_frac'] = ((crops_df['x2'] - crops_df['x1']) * (crops_df['y2'] - crops_df['y1'])) / (
    crops_df['w'] * crops_df['h'])

data.crops_lookup = {(r['image_id'], r['organ']): r for _, r in crops_df.iterrows()}


def has_crop(row):
    return (row['image_id'], row['organ']) in data.crops_lookup


def get_crop(row, crop_row, pad_frac=0.05):
    img = Image.open(row['abs_path']).convert('RGB')
    if not crop_row['found']:
        return img
    w, h = img.size
    x1, y1, x2, y2 = crop_row['x1'], crop_row['y1'], crop_row['x2'], crop_row['y2']
    pad_x, pad_y = (x2 - x1) * pad_frac, (y2 - y1) * pad_frac
    x1, y1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
    x2, y2 = min(w, x2 + pad_x), min(h, y2 + pad_y)
    return img.crop((x1, y1, x2, y2))


MIN_AREA_FRAC = 0.15  # tune based on the histograms/pass-rates below


def has_good_crop(row):
    crop_row = data.crops_lookup.get((row['image_id'], row['organ']))
    if crop_row is None or not crop_row['found']:
        return False
    return crop_row['area_frac'] >= MIN_AREA_FRAC


fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, organ in zip(axes, ORGANS):
    found = crops_df[(crops_df['organ'] == organ) & crops_df['found']]
    ax.hist(found['area_frac'], bins=20, range=(0, 1))
    ax.axvline(MIN_AREA_FRAC, color='red', linestyle='--', label=f'MIN_AREA_FRAC={MIN_AREA_FRAC}')
    ax.set_title(organ)
    ax.set_xlabel('box area / image area')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

for organ in ORGANS:
    sub = data.index[data.index['organ'] == organ]
    n_total = len(sub)
    n_found = sub.apply(has_crop, axis=1).sum()
    n_good = sub.apply(has_good_crop, axis=1).sum()
    print(f'{organ}: {n_total} total, {n_found} with a detection ({n_found / n_total:.1%}), '
          f'{n_good} pass MIN_AREA_FRAC={MIN_AREA_FRAC} ({n_good / n_total:.1%})')


## Step 7: visualize crops

Original image (with the detected box overlaid) next to the resulting crop, for a few "good" crops per organ (passing Step 6's `MIN_AREA_FRAC` filter) and a few that got filtered out (organ detected but too small in frame) - to sanity-check that the filter is doing something sensible. Plus a dedicated look at the two leaf-margin failure cases from `finetune_colab.ipynb` Step 12 - did isolating the leaf actually capture the distinguishing margin/outline?


In [ ]:
def show_pairs(rows, title=''):
    n = len(rows)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    if n == 1:
        axes = axes.reshape(2, 1)
    for j, row in enumerate(rows):
        crop_row = data.crops_lookup[(row['image_id'], row['organ'])]
        img = Image.open(row['abs_path']).convert('RGB')
        axes[0, j].imshow(img)
        if crop_row['found']:
            x1, y1, x2, y2 = crop_row['x1'], crop_row['y1'], crop_row['x2'], crop_row['y2']
            axes[0, j].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                                    fill=False, edgecolor='lime', linewidth=2))
        axes[0, j].set_title(f"{data.species_to_name[row['species_id']]}\narea_frac={crop_row['area_frac']:.2f}", fontsize=8)
        axes[0, j].axis('off')
        axes[1, j].imshow(get_crop(row, crop_row))
        axes[1, j].set_title('crop' if crop_row['found'] else 'NOT FOUND (full image)', fontsize=8)
        axes[1, j].axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


for organ in ORGANS:
    good = data.index[(data.index['organ'] == organ) & data.index.apply(has_good_crop, axis=1)]
    if not good.empty:
        rows = [r for _, r in good.sample(min(4, len(good)), random_state=1).iterrows()]
        show_pairs(rows, title=f'{organ}: good crops (pass MIN_AREA_FRAC)')

    found_mask = data.index.apply(has_crop, axis=1)
    good_mask = data.index.apply(has_good_crop, axis=1)
    filtered_out = data.index[(data.index['organ'] == organ) & found_mask & ~good_mask]
    if not filtered_out.empty:
        rows = [r for _, r in filtered_out.sample(min(4, len(filtered_out)), random_state=1).iterrows()]
        show_pairs(rows, title=f'{organ}: filtered out (organ found but too small in frame)')

# the two known leaf-margin failure cases from finetune_colab.ipynb Step 12
for sp_name in ['Chaerophyllum temulum L.', 'Aegopodium podagraria L.',
                'Althaea officinalis L.', 'Daphne mezereum L.']:
    sub = data.index[(data.index['species_name'] == sp_name) & (data.index['organ'] == 'leaf')]
    sub = sub[sub.apply(has_crop, axis=1)]
    if sub.empty:
        print(f'{sp_name}: no crops available (increase MAX_PER_GROUP / re-run Step 5 for this species)')
        continue
    show_pairs([r for _, r in sub.head(3).iterrows()], title=f'{sp_name} (leaf)')


## Step 8: go/no-go check - Hu-moment shape descriptor on crops

Cheap, CPU-only sanity check before investing in a CNN retrain: recompute the 7-dim log-Hu-moment shape descriptor (Otsu-on-saturation mask, same as `features/leaf.py`'s shape component) on the *crops* instead of the whole image, and run the same z-scored k-NN (k=15, inverse-distance voting) used throughout this project. Compare against the whole-image classical baseline (leaf 0.065/0.183/0.283 etc.) - if cropping doesn't even help this simple descriptor, it's a strong signal the crops aren't isolating the organ well enough yet (revisit `ORGAN_PROMPTS` / `MAX_PER_GROUP` (or loosen `MIN_AREA_FRAC` in Step 6) before sinking time into Step 9+).


In [ ]:
TOP_KS = (1, 5, 10)
K_MATCH = 15

CLASSICAL_BASELINE = {
    'leaf': {'top1': 0.065, 'top5': 0.183, 'top10': 0.283},
    'bark': {'top1': 0.098, 'top5': 0.257, 'top10': 0.383},
    'flower': {'top1': 0.156, 'top5': 0.348, 'top10': 0.472},
}


def hu_moments(img_pil):
    img = np.array(img_pil.convert('RGB'))
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    sat = hsv[:, :, 1]
    _, mask = cv2.threshold(sat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    moments = cv2.moments(mask)
    hu = cv2.HuMoments(moments).flatten()
    return -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)


def knn_topk(emb, sub, standardize=True, k=K_MATCH, top_ks=TOP_KS):
    train_mask = (sub['split'] == 'train').values & sub['use_for_train'].values
    test_mask = (sub['split'] == 'test').values
    gallery, gallery_labels = emb[train_mask], sub['species_id'].values[train_mask]
    query, query_labels = emb[test_mask], sub['species_id'].values[test_mask]
    if standardize:
        mean, std = gallery.mean(0), gallery.std(0)
        std[std < 1e-12] = 1.0
        gallery, query = (gallery - mean) / std, (query - mean) / std
    dists = cdist(query, gallery)
    nn_idx = np.argsort(dists, axis=1)[:, :k]
    hits = {tk: 0 for tk in top_ks}
    for qi, neighbors in enumerate(nn_idx):
        weights = 1.0 / (dists[qi, neighbors] + 1e-6)
        votes = {}
        for sp, w in zip(gallery_labels[neighbors], weights):
            votes[sp] = votes.get(sp, 0.0) + w
        ranked = [sp for sp, _ in sorted(votes.items(), key=lambda x: -x[1])]
        for tk in top_ks:
            if query_labels[qi] in ranked[:tk]:
                hits[tk] += 1
    n = test_mask.sum()
    return {f'top{tk}': hits[tk] / n for tk in top_ks}, n


for organ in ORGANS:
    sub = data.index[(data.index['organ'] == organ) & data.index.apply(has_good_crop, axis=1)].reset_index(drop=True)
    if sub.empty or sub[sub['split'] == 'test'].empty or not sub['use_for_train'].any():
        print(f'{organ}: not enough good crops for a train/test k-NN eval yet, skipping')
        continue
    descs = np.stack([
        hu_moments(get_crop(row, data.crops_lookup[(row['image_id'], row['organ'])]))
        for _, row in tqdm(sub.iterrows(), total=len(sub), desc=f'{organ} hu moments')
    ])
    res, n = knn_topk(descs, sub)
    print(f'{organ} (n_test={n}): hu-on-crop {res}  vs  whole-image classical {CLASSICAL_BASELINE[organ]}')


# free GPU memory - LocateAnything-3B isn't needed past this point
import gc
del locate_model, locate_processor, locate_tokenizer
gc.collect()
torch.cuda.empty_cache()
print('LocateAnything-3B unloaded; GPU memory freed for CNN training in Step 9+. '
      'Re-run Step 3 if you need it again (e.g. to redo Step 5 with a larger MAX_PER_GROUP).')


## Step 9: dataset and transforms for the crop-trained CNN

If Step 8 looked promising (or even just the qualitative crops in Step 7 look like clean leaf/bark/flower isolations), continue here.

**Caveat**: if `MAX_PER_GROUP` was set low in Step 5, `train_df`/`val_df`/`test_df` here are much smaller than the full ~10,777-image dataset used for the whole-image CE baseline - so the comparison below isn't perfectly apples-to-apples (a smaller training set alone would hurt accuracy somewhat, independent of cropping). Treat a result that's *competitive with or better than* the whole-image baseline despite fewer training images as a strong positive signal; treat a small regression cautiously and consider rerunning Step 5 with a larger `MAX_PER_GROUP` before concluding cropping doesn't help.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224
BATCH_SIZE = 64

to_tensor = transforms.ToTensor()
normalize_rgb = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

train_geom = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
])

eval_geom = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
])


class CroppedPlantDataset(Dataset):
    def __init__(self, df, geom):
        self.df = df.reset_index(drop=True)
        self.geom = geom

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        crop_row = data.crops_lookup[(row['image_id'], row['organ'])]
        img = self.geom(get_crop(row, crop_row))
        return normalize_rgb(to_tensor(img)), row['label']


data.crop_index = data.index[data.index.apply(has_good_crop, axis=1)].reset_index(drop=True)
print(f'{len(data.crop_index)} / {len(data.index)} images pass the organ-visibility filter (Step 6)')
print(data.crop_index.groupby(['organ', 'split']).size())

data.train_df = data.crop_index[data.crop_index['use_for_train']]
data.val_df = data.crop_index[data.crop_index['split'] == 'val']
data.test_df = data.crop_index[data.crop_index['split'] == 'test']

val_dl = DataLoader(CroppedPlantDataset(data.val_df, eval_geom), batch_size=BATCH_SIZE,
                     shuffle=False, num_workers=4, pin_memory=True)
test_dl = DataLoader(CroppedPlantDataset(data.test_df, eval_geom), batch_size=BATCH_SIZE,
                      shuffle=False, num_workers=4, pin_memory=True)

print('train/val/test:', len(data.train_df), len(data.val_df), len(data.test_df))


## Step 10: model and training function (CE-only)

Same `mobilenet_v3_small` + cross-entropy setup as `finetune_colab.ipynb`'s CE variant - no SupCon, no edge channel, since this notebook isolates the cropping variable. The projection head is dropped too (only needed for SupCon).


In [ ]:
class EmbeddingClassifier(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        # classifier[0:2] = Linear(576, 1024) + Hardswish -> the 'embedding'
        self.embed = nn.Sequential(backbone.classifier[0], backbone.classifier[1])
        self.embed_dim = backbone.classifier[0].out_features
        self.dropout = backbone.classifier[2]
        self.classifier = nn.Linear(self.embed_dim, n_classes)

    def forward(self, x):
        f = self.features(x)
        f = self.avgpool(f)
        f = torch.flatten(f, 1)
        emb = self.embed(f)
        logits = self.classifier(self.dropout(emb))
        return logits, emb


N_CLASSES = len(data.species_ids)
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
EPOCHS = 15
LR = 3e-4
SEED = 42


def run_training(epochs=EPOCHS, lr=LR, seed=SEED):
    torch.manual_seed(seed)
    model = EmbeddingClassifier(N_CLASSES).to(device)

    train_dl = DataLoader(
        CroppedPlantDataset(data.train_df, train_geom), batch_size=BATCH_SIZE,
        shuffle=True, num_workers=4, pin_memory=True, drop_last=True,
    )

    ce_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler()

    def train_epoch():
        model.train()
        total_loss, total_correct, total_n = 0.0, 0, 0
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.cuda.amp.autocast():
                logits, _ = model(x)
                loss = ce_criterion(logits, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * y.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_n += y.size(0)
        return total_loss / total_n, total_correct / total_n

    @torch.no_grad()
    def eval_epoch(dl):
        model.eval()
        total_loss, total_correct, total_n = 0.0, 0, 0
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.cuda.amp.autocast():
                logits, _ = model(x)
                loss = ce_criterion(logits, y)
            total_loss += loss.item() * y.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_n += y.size(0)
        return total_loss / total_n, total_correct / total_n

    best_val_acc, best_state = 0.0, None
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch()
        val_loss, val_acc = eval_epoch(val_dl)
        scheduler.step()
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'epoch {epoch + 1:2d}/{epochs}  train_loss={train_loss:.3f} train_acc={train_acc:.3f}  '
              f'val_loss={val_loss:.3f} val_acc={val_acc:.3f}  best_val_acc={best_val_acc:.3f}')

    model.load_state_dict(best_state)
    return model, best_val_acc


## Step 11: train CE on crops (~10 min on an A100 for 15 epochs)


In [ ]:
model_ce_crop, val_acc_ce_crop = run_training()
print('CE (crops) best val acc:', val_acc_ce_crop)


## Step 12: test-set top-1/5/10 (classifier head) vs whole-image CE baseline


In [ ]:
@torch.no_grad()
def topk_accuracy(model, dl, df):
    model.eval()
    all_logits = []
    for x, _ in dl:
        x = x.to(device)
        with torch.cuda.amp.autocast():
            logits, _ = model(x)
        all_logits.append(logits.float().cpu())
    logits = torch.cat(all_logits)
    labels = torch.tensor(df['label'].values)
    topk = logits.topk(max(TOP_KS), dim=1).indices
    return {f'top{k}': (topk[:, :k] == labels[:, None]).any(1).float().mean().item() for k in TOP_KS}


BASELINE_CLF_CE = {
    'leaf': {'top1': 0.581, 'top5': 0.877, 'top10': 0.939},
    'bark': {'top1': 0.459, 'top5': 0.716, 'top10': 0.787},
    'flower': {'top1': 0.716, 'top5': 0.921, 'top10': 0.958},
}

classifier_results = {}
for organ in ORGANS:
    sub_df = data.test_df[data.test_df['organ'] == organ]
    if sub_df.empty:
        print(f'{organ}: no test crops, skipping')
        continue
    sub_dl = DataLoader(CroppedPlantDataset(sub_df, eval_geom), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)
    classifier_results[organ] = topk_accuracy(model_ce_crop, sub_dl, sub_df)
    print(f'{organ} (n={len(sub_df)}): crop {classifier_results[organ]}  vs  whole-image CE {BASELINE_CLF_CE[organ]}')


## Step 12b: crop + whole image (dual-input, weighted fusion)

Try using the *whole* (uncropped) image as a second input alongside the crop, on the
hypothesis that surrounding context (overall plant habit, leaf arrangement, etc.) carries
some signal the crop alone discards.

`CropAndWholeDataset` returns both views per sample. `DualEmbeddingClassifier` reuses the
*same* `mobilenet_v3_small` backbone (shared weights) to embed each view, then concatenates
`[crop_emb, WHOLE_IMAGE_WEIGHT * whole_emb]` (2048-dim) before the classifier head.

`WHOLE_IMAGE_WEIGHT` is the knob to play with:
- `0.0` zeroes out the whole-image half of the embedding (effectively crop-only, though the
  classifier head is still 2048-dim, so not identical to Step 9-12's model).
- `1.0` gives both views equal footing.

The weight is baked in *during training* (the classifier head learns to use whatever scale
the embeddings arrive at), so trying a different value means re-running Step 12c/12d - each
run is a fresh ~10-15 min training on an A100.


In [ ]:
# ---- constants -----------------------------------------------------------
WHOLE_IMAGE_WEIGHT = 0.5  # 0 = whole-image half zeroed out, 1 = equal footing with the crop


# ---- dataset ---------------------------------------------------------------
class CropAndWholeDataset(Dataset):
    def __init__(self, df, geom):
        self.df = df.reset_index(drop=True)
        self.geom = geom

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        crop_row = data.crops_lookup[(row['image_id'], row['organ'])]
        crop_img = self.geom(get_crop(row, crop_row))
        whole_img = self.geom(Image.open(row['abs_path']).convert('RGB'))
        return normalize_rgb(to_tensor(crop_img)), normalize_rgb(to_tensor(whole_img)), row['label']


# ---- model -------------------------------------------------------------------
class DualEmbeddingClassifier(nn.Module):
    """Shared mobilenet_v3_small backbone embeds the crop and the whole image;
    embeddings concatenated as [crop_emb, whole_weight * whole_emb] before the classifier head."""

    def __init__(self, n_classes):
        super().__init__()
        backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        self.embed = nn.Sequential(backbone.classifier[0], backbone.classifier[1])
        self.embed_dim = backbone.classifier[0].out_features
        self.dropout = backbone.classifier[2]
        self.classifier = nn.Linear(2 * self.embed_dim, n_classes)

    def embed_view(self, x):
        f = self.features(x)
        f = self.avgpool(f)
        f = torch.flatten(f, 1)
        return self.embed(f)

    def forward(self, x_crop, x_whole, whole_weight=WHOLE_IMAGE_WEIGHT):
        crop_emb = self.embed_view(x_crop)
        whole_emb = self.embed_view(x_whole)
        combined = torch.cat([crop_emb, whole_weight * whole_emb], dim=1)
        logits = self.classifier(self.dropout(combined))
        return logits, combined


# ---- training function -------------------------------------------------------
def run_training_dual(epochs=EPOCHS, lr=LR, seed=SEED, whole_image_weight=WHOLE_IMAGE_WEIGHT):
    torch.manual_seed(seed)
    model = DualEmbeddingClassifier(N_CLASSES).to(device)

    train_dl = DataLoader(
        CropAndWholeDataset(data.train_df, train_geom), batch_size=BATCH_SIZE,
        shuffle=True, num_workers=4, pin_memory=True, drop_last=True,
    )
    val_dl_dual = DataLoader(CropAndWholeDataset(data.val_df, eval_geom), batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)

    ce_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler()

    def train_epoch():
        model.train()
        total_loss, total_correct, total_n = 0.0, 0, 0
        for x_crop, x_whole, y in train_dl:
            x_crop = x_crop.to(device, non_blocking=True)
            x_whole = x_whole.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.cuda.amp.autocast():
                logits, _ = model(x_crop, x_whole, whole_image_weight)
                loss = ce_criterion(logits, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * y.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_n += y.size(0)
        return total_loss / total_n, total_correct / total_n

    @torch.no_grad()
    def eval_epoch(dl):
        model.eval()
        total_loss, total_correct, total_n = 0.0, 0, 0
        for x_crop, x_whole, y in dl:
            x_crop = x_crop.to(device, non_blocking=True)
            x_whole = x_whole.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.cuda.amp.autocast():
                logits, _ = model(x_crop, x_whole, whole_image_weight)
                loss = ce_criterion(logits, y)
            total_loss += loss.item() * y.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_n += y.size(0)
        return total_loss / total_n, total_correct / total_n

    best_val_acc, best_state = 0.0, None
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch()
        val_loss, val_acc = eval_epoch(val_dl_dual)
        scheduler.step()
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'epoch {epoch + 1:2d}/{epochs}  train_loss={train_loss:.3f} train_acc={train_acc:.3f}  '
              f'val_loss={val_loss:.3f} val_acc={val_acc:.3f}  best_val_acc={best_val_acc:.3f}')

    model.load_state_dict(best_state)
    return model, best_val_acc


## Step 12c: train crop + whole-image model (~10-15 min on an A100 for 15 epochs)


In [ ]:
model_ce_dual, val_acc_ce_dual = run_training_dual()
print(f'CE (crop + whole image, weight={WHOLE_IMAGE_WEIGHT}) best val acc:', val_acc_ce_dual)


## Step 12d: test-set top-1/5/10 vs crop-only (Step 12) and whole-image baselines


In [ ]:
@torch.no_grad()
def topk_accuracy_dual(model, dl, df, whole_image_weight=WHOLE_IMAGE_WEIGHT):
    model.eval()
    all_logits = []
    for x_crop, x_whole, _ in dl:
        x_crop, x_whole = x_crop.to(device), x_whole.to(device)
        with torch.cuda.amp.autocast():
            logits, _ = model(x_crop, x_whole, whole_image_weight)
        all_logits.append(logits.float().cpu())
    logits = torch.cat(all_logits)
    labels = torch.tensor(df['label'].values)
    topk = logits.topk(max(TOP_KS), dim=1).indices
    return {f'top{k}': (topk[:, :k] == labels[:, None]).any(1).float().mean().item() for k in TOP_KS}


def make_eval_loader_dual(df, batch_size=BATCH_SIZE):
    return DataLoader(CropAndWholeDataset(df, eval_geom), batch_size=batch_size, shuffle=False, num_workers=4)


classifier_results_dual = {}
for organ in ORGANS:
    sub_df = data.test_df[data.test_df['organ'] == organ]
    if sub_df.empty:
        print(f'{organ}: no test crops, skipping')
        continue
    sub_dl = make_eval_loader_dual(sub_df)
    classifier_results_dual[organ] = topk_accuracy_dual(model_ce_dual, sub_dl, sub_df)
    print(f'{organ} (n={len(sub_df)}): crop+whole (w={WHOLE_IMAGE_WEIGHT}) {classifier_results_dual[organ]}  '
          f'vs  crop-only {classifier_results[organ]}  vs  whole-image CE {BASELINE_CLF_CE[organ]}')


## Step 13: extract embeddings + in-notebook k-NN comparison vs whole-image baseline

Reuses `knn_topk` from Step 8 (z-scored, k=15, inverse-distance voting) - now applied to the CNN embedding instead of Hu moments.

`ORGANS`, `make_eval_loader`, and `organ_subset` are defined here and reused by Steps 14-15 below.


In [ ]:
# ---- constants -----------------------------------------------------------
BASELINE_KNN_CE = {
    'leaf': {'top1': 0.517, 'top5': 0.830, 'top10': 0.882},
    'bark': {'top1': 0.317, 'top5': 0.579, 'top10': 0.672},
    'flower': {'top1': 0.679, 'top5': 0.894, 'top10': 0.926},
}


# ---- functions -------------------------------------------------------------
@torch.no_grad()
def embed_all(model, dl):
    model.eval()
    embs = []
    for x, _ in dl:
        x = x.to(device)
        with torch.cuda.amp.autocast():
            _, emb = model(x)
        embs.append(emb.float().cpu().numpy())
    return np.concatenate(embs)


def make_eval_loader(df, batch_size=BATCH_SIZE):
    return DataLoader(CroppedPlantDataset(df, eval_geom), batch_size=batch_size, shuffle=False, num_workers=4)


def organ_subset(df, emb, organ):
    sub = df[df['organ'] == organ].reset_index(drop=True)
    organ_emb = emb[(df['organ'] == organ).values]
    return sub, organ_emb


def compare_knn_to_baseline(emb, df, organ, baseline):
    sub, organ_emb = organ_subset(df, emb, organ)
    if sub.empty or sub[sub['split'] == 'test'].empty or not sub['use_for_train'].any():
        print(f'{organ}: not enough crops for k-NN eval, skipping')
        return None
    res, n = knn_topk(organ_emb, sub)
    print(f'{organ} (n_test={n}): k-NN crop {res}  vs  whole-image k-NN ce_emb {baseline[organ]}')
    return res


# ---- run ---------------------------------------------------------------------
all_dl = make_eval_loader(data.crop_index)
ce_crop_emb = embed_all(model_ce_crop, all_dl)
print('ce_crop_emb', ce_crop_emb.shape)

knn_results = {}
for organ in ORGANS:
    res = compare_knn_to_baseline(ce_crop_emb, data.crop_index, organ, BASELINE_KNN_CE)
    if res is not None:
        knn_results[organ] = res


## Step 14: qualitative inspection - classifier successes and failures (on crops)

Same format as `finetune_colab.ipynb` Step 11, but images shown are the *crops* fed to the model.


In [ ]:
# ---- constants -----------------------------------------------------------
GRID_N = 8
GRID_NCOLS = 4
PRED_K = 5
PRED_SHOW_K = 3


# ---- functions -------------------------------------------------------------
@torch.no_grad()
def predict_topk(model, dl, k=PRED_K):
    model.eval()
    all_probs = []
    for x, _ in dl:
        x = x.to(device)
        with torch.cuda.amp.autocast():
            logits, _ = model(x)
        all_probs.append(F.softmax(logits.float(), dim=1).cpu())
    return torch.cat(all_probs).topk(k, dim=1)


def format_predictions(topk_idx_row, topk_probs_row, idx_to_species, n=PRED_SHOW_K):
    preds = [(data.species_to_name[idx_to_species[j]], p) for j, p in zip(topk_idx_row[:n], topk_probs_row[:n])]
    return '\n'.join(f'{name} ({p:.2f})' for name, p in preds)


def plot_grid(df, indices, topk_idx, topk_probs, idx_to_species, title=''):
    nrows = -(-len(indices) // GRID_NCOLS)
    fig, axes = plt.subplots(nrows, GRID_NCOLS, figsize=(4 * GRID_NCOLS, 4 * nrows))
    axes = np.atleast_1d(axes).flatten()
    for ax, i in zip(axes, indices):
        row = df.iloc[i]
        ax.imshow(get_crop(row, data.crops_lookup[(row['image_id'], row['organ'])]))
        true_name = data.species_to_name[row['species_id']]
        pred_str = format_predictions(topk_idx[i], topk_probs[i], idx_to_species)
        ax.set_title(f'true: {true_name}\n{pred_str}', fontsize=8)
        ax.axis('off')
    for ax in axes[len(indices):]:
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_grid(df, mask, organ, topk_idx, topk_probs, idx_to_species, n=GRID_N, title=''):
    organ_mask = (df['organ'] == organ).values
    candidates = np.where(mask & organ_mask)[0]
    if len(candidates) == 0:
        print(f'no examples: {title}')
        return
    chosen = np.random.choice(candidates, size=min(n, len(candidates)), replace=False)
    plot_grid(df, chosen, topk_idx, topk_probs, idx_to_species, title=title)


def run_qualitative_grids(df, topk_idx, topk_probs, correct, idx_to_species):
    for organ in ORGANS:
        organ_mask = (df['organ'] == organ).values
        if not organ_mask.any():
            continue
        organ_acc = correct[organ_mask].mean()
        print(f'=== {organ}: CE(crop) classifier top-1 test accuracy = {organ_acc:.3f} ===')
        show_grid(df, correct, organ, topk_idx, topk_probs, idx_to_species, title=f'{organ}: correct (top-3 shown)')
        show_grid(df, ~correct, organ, topk_idx, topk_probs, idx_to_species, title=f'{organ}: INCORRECT (top-3 shown)')


# ---- run ---------------------------------------------------------------------
idx_to_species = {i: sp for sp, i in data.species_to_idx.items()}
test_df_r = data.test_df.reset_index(drop=True)
test_dl_r = make_eval_loader(test_df_r)
topk_probs, topk_idx = predict_topk(model_ce_crop, test_dl_r)
topk_probs, topk_idx = topk_probs.numpy(), topk_idx.numpy()
correct = topk_idx[:, 0] == test_df_r['label'].values

run_qualitative_grids(test_df_r, topk_idx, topk_probs, correct, idx_to_species)


## Step 15: qualitative inspection - k-NN retrieval (on crops), revisiting the known failure cases

Same format as `finetune_colab.ipynb` Step 12, on crop embeddings, plus a dedicated re-check of the *Chaerophyllum temulum* and *Althaea officinalis* leaf queries that were misretrieved before - do their nearest neighbors change once the leaf outline is isolated from the background?


In [ ]:
# ---- constants -----------------------------------------------------------
KNN_VIZ_K = 5
N_CORRECT_EXAMPLES = 2
N_INCORRECT_EXAMPLES = 2
VIZ_SEED = 0
FAILURE_CASE_SPECIES = ['Chaerophyllum temulum L.', 'Althaea officinalis L.']
N_FAILURE_EXAMPLES = 2


# ---- functions -------------------------------------------------------------
def split_gallery_query(emb, organ):
    sub, organ_emb = organ_subset(data.crop_index, emb, organ)
    train_mask = (sub['split'] == 'train').values & sub['use_for_train'].values
    test_mask = (sub['split'] == 'test').values
    gallery, gallery_labels = organ_emb[train_mask], sub['species_id'].values[train_mask]
    gallery_rows = sub[train_mask].reset_index(drop=True)
    query, query_labels = organ_emb[test_mask], sub['species_id'].values[test_mask]
    query_rows = sub[test_mask].reset_index(drop=True)
    return gallery, gallery_labels, gallery_rows, query, query_labels, query_rows


def zscore_stats(gallery):
    mean, std = gallery.mean(0), gallery.std(0)
    std[std < 1e-12] = 1.0
    return mean, std


def knn_predict(query_n, gallery_n, gallery_labels, k=KNN_VIZ_K):
    dists = cdist(query_n, gallery_n)
    nn_idx = np.argsort(dists, axis=1)[:, :k]
    pred_sp = []
    for qi, neighbors in enumerate(nn_idx):
        w = 1.0 / (dists[qi, neighbors] + 1e-6)
        votes = {}
        for sp, wi in zip(gallery_labels[neighbors], w):
            votes[sp] = votes.get(sp, 0.0) + wi
        pred_sp.append(max(votes.items(), key=lambda x: x[1])[0])
    return dists, nn_idx, np.array(pred_sp)


def plot_query_with_neighbors(query_row, query_title, dists_row, nn_idx_row, gallery_rows, gallery_labels):
    k = len(nn_idx_row)
    fig, axes = plt.subplots(1, k + 1, figsize=(3 * (k + 1), 3.2))
    axes[0].imshow(get_crop(query_row, data.crops_lookup[(query_row['image_id'], query_row['organ'])]))
    axes[0].set_title(query_title, fontsize=8)
    axes[0].axis('off')
    for j, ni in enumerate(nn_idx_row):
        g_row = gallery_rows.iloc[ni]
        axes[j + 1].imshow(get_crop(g_row, data.crops_lookup[(g_row['image_id'], g_row['organ'])]))
        axes[j + 1].set_title(f'NN{j+1} d={dists_row[ni]:.1f}\n{data.species_to_name[gallery_labels[ni]]}', fontsize=8)
        axes[j + 1].axis('off')
    plt.tight_layout()
    plt.show()


def knn_visual_examples(embeddings, organ, standardize=True, k=KNN_VIZ_K,
                         n_correct=N_CORRECT_EXAMPLES, n_incorrect=N_INCORRECT_EXAMPLES, seed=VIZ_SEED):
    rng = np.random.default_rng(seed)
    gallery, gallery_labels, gallery_rows, query, query_labels, query_rows = split_gallery_query(embeddings, organ)

    mean, std = zscore_stats(gallery)
    if standardize:
        gallery_n, query_n = (gallery - mean) / std, (query - mean) / std
    else:
        gallery_n, query_n = gallery, query

    dists, nn_idx, pred_sp = knn_predict(query_n, gallery_n, gallery_labels, k=k)
    correct = pred_sp == query_labels

    for label, mask, n in (('CORRECT', correct, n_correct), ('INCORRECT', ~correct, n_incorrect)):
        candidates = np.where(mask)[0]
        if len(candidates) == 0:
            continue
        for qi in rng.choice(candidates, size=min(n, len(candidates)), replace=False):
            title = (f'QUERY ({label})\ntrue: {data.species_to_name[query_labels[qi]]}\n'
                     f'pred: {data.species_to_name[pred_sp[qi]]}')
            plot_query_with_neighbors(query_rows.iloc[qi], title, dists[qi], nn_idx[qi], gallery_rows, gallery_labels)

    return gallery, gallery_labels, gallery_rows, mean, std


def show_failure_case_neighbors(species_names, embeddings, organ, gallery, gallery_labels, gallery_rows, mean, std,
                                 k=KNN_VIZ_K, n=N_FAILURE_EXAMPLES):
    sub, emb = organ_subset(data.crop_index, embeddings, organ)
    gallery_n = (gallery - mean) / std
    for sp_name in species_names:
        q_rows = sub[(sub['species_name'] == sp_name) & (sub['split'] == 'test')]
        if q_rows.empty:
            print(f'{sp_name}: no test crops available')
            continue
        for qi in q_rows.head(n).index:
            q_row = sub.iloc[qi]
            q_n = (emb[qi] - mean) / std
            dists, nn_idx, _ = knn_predict(q_n[None, :], gallery_n, gallery_labels, k=k)
            title = f'QUERY\ntrue: {sp_name}'
            plot_query_with_neighbors(q_row, title, dists[0], nn_idx[0], gallery_rows, gallery_labels)


# ---- run ---------------------------------------------------------------------
gallery_cache = {}
for organ in ORGANS:
    sub, _ = organ_subset(data.crop_index, ce_crop_emb, organ)
    if sub.empty or sub[sub['split'] == 'test'].empty or not sub['use_for_train'].any():
        print(f'{organ}: not enough crops, skipping')
        continue
    print(f'=== {organ}: k-NN retrieval examples (CE on crops) ===')
    gallery_cache[organ] = knn_visual_examples(ce_crop_emb, organ)

if 'leaf' in gallery_cache:
    print('=== revisiting known failure cases (leaf, CE on crops) ===')
    gallery, gallery_labels, gallery_rows, mean, std = gallery_cache['leaf']
    show_failure_case_neighbors(FAILURE_CASE_SPECIES, ce_crop_emb, 'leaf', gallery, gallery_labels, gallery_rows, mean, std)


## Step 15b: per-species crop coverage (diagnostic, needs only Step 9)

Before comparing any accuracy numbers: how many *training* crops does each
species actually have? `MAX_PER_GROUP` + `MIN_AREA_FRAC` compound, so a species
can end up with **zero** training crops while still occupying a slot in the
87-way softmax and still appearing in the test set - mechanically unlearnable,
and a purely artifactual contribution to the accuracy drop.

Cheap, CPU-only, no model needed.


In [ ]:
# ---- per-species crop coverage ----------------------------------------------
coverage_rows = []
for organ in ORGANS:
    full = data.index[data.index['organ'] == organ]
    crops = data.crop_index[data.crop_index['organ'] == organ]

    train_counts = (crops[crops['use_for_train']]
                    .groupby('species_id').size()
                    .reindex(data.species_ids, fill_value=0))
    test_counts = (crops[crops['split'] == 'test']
                   .groupby('species_id').size()
                   .reindex(data.species_ids, fill_value=0))

    # fraction of each species' original training images that survived
    # MAX_PER_GROUP + MIN_AREA_FRAC
    full_train_counts = (full[full['use_for_train']]
                         .groupby('species_id').size()
                         .reindex(data.species_ids, fill_value=0))
    survival = train_counts / full_train_counts.replace(0, np.nan)

    # species that appear in the crop test set but have no crop to learn from
    unlearnable = ((train_counts == 0) & (test_counts > 0))
    n_test_unlearnable = int(test_counts[unlearnable].sum())
    n_test_total = int(test_counts.sum())

    coverage_rows.append({
        'organ': organ,
        'species_total': len(data.species_ids),
        'species_with_train_crops': int((train_counts > 0).sum()),
        'species_zero_train_crops': int((train_counts == 0).sum()),
        'species_unlearnable_but_tested': int(unlearnable.sum()),
        'test_imgs_unlearnable': n_test_unlearnable,
        'test_imgs_total': n_test_total,
        'test_frac_unlearnable': (n_test_unlearnable / n_test_total) if n_test_total else 0.0,
        'median_train_crops_per_species': float(train_counts.median()),
        'min_train_crops_per_species': int(train_counts.min()),
        # how hard did the filter thin each species? uniform thinning vs.
        # concentration on particular growth forms is the mechanism question.
        'median_train_survival_frac': float(survival.median()),
        'min_train_survival_frac': float(survival.min()),
    })

per_species_coverage = pd.DataFrame(coverage_rows).set_index('organ')
print(per_species_coverage.T.to_string())
print()
for r in coverage_rows:
    print(f"{r['organ']}: {r['species_zero_train_crops']}/{r['species_total']} species have no training crop; "
          f"{r['species_unlearnable_but_tested']} of those still appear in the crop test set, "
          f"accounting for {r['test_imgs_unlearnable']}/{r['test_imgs_total']} test images "
          f"({r['test_frac_unlearnable']:.1%} - a top-1 accuracy ceiling of "
          f"{1 - r['test_frac_unlearnable']:.1%} for this organ; top-5/top-10 are unaffected).")
    print(f"    training images surviving MAX_PER_GROUP + MIN_AREA_FRAC: "
          f"median {r['median_train_survival_frac']:.1%} of the original per species, "
          f"min {r['min_train_survival_frac']:.1%} - a low min means the filter "
          f"concentrated on particular species rather than thinning uniformly.")


## Step 15c: whole-image-weight ablation on the **already-trained** dual model

No retraining. `DualEmbeddingClassifier.forward` takes `whole_weight` at call
time, so the trained `model_ce_dual` can be scored with the whole-image half
scaled from 0 (crop only) to 1 (equal footing) - one model, one test set, one
variable.

**Caveat**: the model was trained at `WHOLE_IMAGE_WEIGHT=0.5`, so `w != 0.5` is
off-distribution for the classifier head. Read the curve as directional evidence
about how much the head leans on the whole-image half, not as a clean
crop-vs-whole measurement.

If your Colab runtime from the previous run is **still alive**, you can paste
just this cell in and get the ablation for free before the runtime recycles.


In [ ]:
# ---- constants ---------------------------------------------------------------
ABLATION_WEIGHTS = (0.0, 0.25, 0.5, 0.75, 1.0)


# ---- run ---------------------------------------------------------------------
dual_weight_ablation = {}
for organ in ORGANS:
    sub_df = data.test_df[data.test_df['organ'] == organ]
    if sub_df.empty:
        print(f'{organ}: no test crops, skipping')
        continue
    sub_dl = make_eval_loader_dual(sub_df)  # built once, reused across weights
    dual_weight_ablation[organ] = {}
    for w in ABLATION_WEIGHTS:
        res = topk_accuracy_dual(model_ce_dual, sub_dl, sub_df, whole_image_weight=w)
        dual_weight_ablation[organ][str(w)] = res
        print(f'{organ} (n={len(sub_df)})  w={w:.2f}  {res}')
    print()

print('top-1 by whole-image weight:')
print('  ' + 'organ '.ljust(8) + ''.join(f'w={w:<8.2f}' for w in ABLATION_WEIGHTS))
for organ, by_w in dual_weight_ablation.items():
    cells = ''.join(f'{by_w[str(w)]["top1"]:<10.3f}' for w in ABLATION_WEIGHTS)
    print('  ' + organ.ljust(8) + cells)


## Step 15d: the matched control - same images, same splits, **uncropped**

The Step 12/13 tables compare crop-trained models against `BASELINE_CLF_CE` /
`BASELINE_KNN_CE`, which were measured in `finetune_colab.ipynb` on a different
and much larger population. Three things differ at once there, and they don't
push the same way:

1. **training set** - `MAX_PER_GROUP=20` + `MIN_AREA_FRAC` leave ~0.25x (leaf) /
   0.20x (flower) / 0.73x (bark) of the baseline's training images;
2. **test set** - `MIN_AREA_FRAC` filters *every* split, so the crop models were
   scored on 131/134/110 images vs the baseline's 755/183/686 (17% / 73% / 16%).
   That surviving subpopulation is the closer-framed, larger-in-frame photos,
   i.e. plausibly *easier*;
3. the input actually being cropped - the only variable of interest.

This cell removes (1) and (2): train the identical model on the **identical
`image_id`s and splits**, with the images uncropped. Everything else -
architecture, seed, epochs, LR, schedule, augmentation, batch size - is held
fixed. ~10 min on an A100, no VLM calls.

`run_training_generic` is Step 10's `run_training` with the dataset class
parameterized; `run_training_generic(CroppedPlantDataset)` should land within
noise of the crop-only `best_val_acc` (0.349 in the previous run) - a cheap
integrity check if you want it, at the cost of another ~10 min.


In [ ]:
# ---- dataset: same rows, no cropping -----------------------------------------
class WholeImagePlantDataset(Dataset):
    """Same interface as CroppedPlantDataset, but ignores the bounding box."""

    def __init__(self, df, geom):
        self.df = df.reset_index(drop=True)
        self.geom = geom

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = self.geom(Image.open(row['abs_path']).convert('RGB'))
        return normalize_rgb(to_tensor(img)), row['label']


# ---- training: Step 10's run_training, dataset class parameterized -----------
def run_training_generic(dataset_cls, epochs=EPOCHS, lr=LR, seed=SEED):
    torch.manual_seed(seed)
    model = EmbeddingClassifier(N_CLASSES).to(device)

    train_dl = DataLoader(
        dataset_cls(data.train_df, train_geom), batch_size=BATCH_SIZE,
        shuffle=True, num_workers=4, pin_memory=True, drop_last=True,
    )
    val_dl_generic = DataLoader(
        dataset_cls(data.val_df, eval_geom), batch_size=BATCH_SIZE,
        shuffle=False, num_workers=4, pin_memory=True,
    )

    ce_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler()

    def run_epoch(dl, train):
        model.train() if train else model.eval()
        total_loss, total_correct, total_n = 0.0, 0, 0
        with torch.set_grad_enabled(train):
            for x, y in dl:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                with torch.cuda.amp.autocast():
                    logits, _ = model(x)
                    loss = ce_criterion(logits, y)
                if train:
                    optimizer.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                total_loss += loss.item() * y.size(0)
                total_correct += (logits.argmax(1) == y).sum().item()
                total_n += y.size(0)
        return total_loss / total_n, total_correct / total_n

    best_val_acc, best_state = 0.0, None
    for epoch in range(epochs):
        train_loss, train_acc = run_epoch(train_dl, train=True)
        val_loss, val_acc = run_epoch(val_dl_generic, train=False)
        scheduler.step()
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'epoch {epoch + 1:2d}/{epochs}  train_loss={train_loss:.3f} train_acc={train_acc:.3f}  '
              f'val_loss={val_loss:.3f} val_acc={val_acc:.3f}  best_val_acc={best_val_acc:.3f}')

    model.load_state_dict(best_state)
    return model, best_val_acc


model_ce_whole, val_acc_ce_whole = run_training_generic(WholeImagePlantDataset)
print('CE (whole image, matched subset) best val acc:', val_acc_ce_whole,
      '  vs crop-only:', val_acc_ce_crop)


### Step 15e: matched comparison - classifier head and k-NN

Both arms now share `data.train_df` / `data.val_df` / `data.test_df`, so these
are like-for-like. The `BASELINE_*` columns are retained only as a reminder of
the full-data ceiling - they are *not* the control.


In [ ]:
# ---- classifier head ---------------------------------------------------------
def make_eval_loader_whole(df, batch_size=BATCH_SIZE):
    return DataLoader(WholeImagePlantDataset(df, eval_geom), batch_size=batch_size,
                      shuffle=False, num_workers=4)


classifier_results_whole_matched = {}
for organ in ORGANS:
    sub_df = data.test_df[data.test_df['organ'] == organ]
    if sub_df.empty:
        print(f'{organ}: no test crops, skipping')
        continue
    res = topk_accuracy(model_ce_whole, make_eval_loader_whole(sub_df), sub_df)
    classifier_results_whole_matched[organ] = res
    print(f'{organ} (n={len(sub_df)}): whole-matched {res}  vs  crop {classifier_results[organ]}')

# ---- k-NN over embeddings ----------------------------------------------------
whole_matched_emb = embed_all(model_ce_whole, make_eval_loader_whole(data.crop_index))
print('whole_matched_emb', whole_matched_emb.shape)

knn_results_whole_matched = {}
for organ in ORGANS:
    sub, organ_emb = organ_subset(data.crop_index, whole_matched_emb, organ)
    if sub.empty or sub[sub['split'] == 'test'].empty or not sub['use_for_train'].any():
        print(f'{organ}: not enough images for k-NN eval, skipping')
        continue
    res, n = knn_topk(organ_emb, sub)
    knn_results_whole_matched[organ] = res
    print(f'{organ} (n_test={n}): k-NN whole-matched {res}  vs  k-NN crop {knn_results[organ]}')

# ---- headline table ----------------------------------------------------------
print()
print('MATCHED COMPARISON (same image_ids, same splits) - top-1')
print(f"{'organ':<8}{'crop':>10}{'whole':>10}{'delta':>10}   | full-data whole-image baseline")
for organ in ORGANS:
    if organ not in classifier_results_whole_matched:
        continue
    c = classifier_results[organ]['top1']
    w = classifier_results_whole_matched[organ]['top1']
    print(f'{organ:<8}{c:>10.3f}{w:>10.3f}{w - c:>+10.3f}   | {BASELINE_CLF_CE[organ]["top1"]:.3f}')
print()
print('  delta > 0  ->  cropping HURTS: the crop discards usable signal')
print('  delta ~ 0  ->  cropping is a no-op on this subset (consistent with')
print('                 the bimodal area_frac: most boxes are ~the whole frame)')
print('  delta < 0  ->  cropping HELPS, and the Step 12/13 drop was entirely')
print('                 the train/test population change')
print()
print('  NB: this is matched WITHIN the MIN_AREA_FRAC subset (16% / 73% / 16% of')
print('      the original leaf/bark/flower test sets). delta < 0 means cropping')
print('      helps *where grounding yields a usable box* - not that cropping is')
print('      deployable corpus-wide. A MAX_PER_GROUP=None rerun would not change')
print('      that: the limit is the bimodal area_frac (44% of flower images have')
print('      no usable box at any sample size), not the per-group cap.')


## Step 16: export

Exports `organ_crops.parquet` (bounding boxes for whatever subset Step 5 processed - useful locally even if the CNN comparison above is inconclusive), the crop-trained CE embeddings/model/ONNX, and a `metadata.json` with the comparison tables against the whole-image baselines.


In [ ]:
!pip install -q onnx

OUT = Path('/content/locate_crop_export')
OUT.mkdir(exist_ok=True)

data.crops_df.to_parquet(OUT / 'organ_crops.parquet')

for organ in ORGANS:
    mask = (data.crop_index['organ'] == organ).values
    if not mask.any():
        continue
    np.savez_compressed(
        OUT / f'descriptors_{organ}_ce_crop_emb.npz',
        descriptor=ce_crop_emb[mask],
        image_id=data.crop_index['image_id'].values[mask],
        species_id=data.crop_index['species_id'].values[mask],
        species_name=data.crop_index['species_name'].values[mask],
        split=data.crop_index['split'].values[mask],
    )

if 'whole_matched_emb' in globals():
    for organ in ORGANS:
        mask = (data.crop_index['organ'] == organ).values
        if not mask.any():
            continue
        np.savez_compressed(
            OUT / f'descriptors_{organ}_ce_whole_matched_emb.npz',
            descriptor=whole_matched_emb[mask],
            image_id=data.crop_index['image_id'].values[mask],
            species_id=data.crop_index['species_id'].values[mask],
            species_name=data.crop_index['species_name'].values[mask],
            split=data.crop_index['split'].values[mask],
        )

torch.save(model_ce_crop.state_dict(), OUT / 'mobilenet_v3_small_ce_crop.pt')
if 'model_ce_whole' in globals():
    torch.save(model_ce_whole.state_dict(), OUT / 'mobilenet_v3_small_ce_whole_matched.pt')

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
torch.onnx.export(
    model_ce_crop, dummy, str(OUT / 'mobilenet_v3_small_ce_crop.onnx'),
    input_names=['image'], output_names=['logits', 'embedding'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17, dynamo=False,
)

metadata = {
    'max_per_group': MAX_PER_GROUP,
    'organ_prompts': ORGAN_PROMPTS,
    'min_area_frac': MIN_AREA_FRAC,
    'good_crop_rate_among_detections': {
        organ: float(data.crops_df[(data.crops_df['organ'] == organ) & data.crops_df['found']]['area_frac']
                      .ge(MIN_AREA_FRAC).mean())
        for organ in ORGANS
    },
    'n_crops': {organ: int((data.crop_index['organ'] == organ).sum()) for organ in ('leaf', 'bark', 'flower')},
    'found_rate': data.crops_df.groupby('organ')['found'].mean().to_dict(),
    'classifier_head': {'ce_crop': classifier_results, 'baseline_ce_whole_image': BASELINE_CLF_CE},
    'knn_comparison': {'ce_crop_emb': knn_results, 'baseline_ce_emb_whole_image': BASELINE_KNN_CE},
    'best_val_acc_ce_crop': val_acc_ce_crop,
    # ---- matched control + ablations (Steps 15b-15e); omitted if not run ----
    'per_species_coverage': (per_species_coverage.reset_index().to_dict('records')
                             if 'per_species_coverage' in globals() else None),
    'dual_weight_ablation': globals().get('dual_weight_ablation'),
    'classifier_head_whole_matched': globals().get('classifier_results_whole_matched'),
    'knn_whole_matched': globals().get('knn_results_whole_matched'),
    'best_val_acc_ce_whole_matched': globals().get('val_acc_ce_whole'),
    'classifier_head_dual': globals().get('classifier_results_dual'),
    'best_val_acc_ce_dual': globals().get('val_acc_ce_dual'),
    'whole_image_weight': globals().get('WHOLE_IMAGE_WEIGHT'),
    'embed_dim': int(ce_crop_emb.shape[1]),
    'epochs': EPOCHS,
    'img_size': IMG_SIZE,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std': IMAGENET_STD,
}
def _jsonable(o):
    """numpy scalars (np.int64 in particular) aren't JSON-serializable natively."""
    if hasattr(o, 'item'):
        return o.item()
    raise TypeError(f'{type(o)} is not JSON serializable')


with open(OUT / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=_jsonable)

print(json.dumps(metadata, indent=2, default=_jsonable))


In [ ]:
import shutil
shutil.make_archive('/content/locate_crop_export', 'zip', OUT)
files.download('/content/locate_crop_export.zip')


## Next steps (back in the local `plantid` repo)

1. Extract `locate_crop_export.zip`. `metadata.json` has the headline comparison: crop-trained CE classifier-head/k-NN numbers vs the whole-image CE baseline, per organ.
2. `organ_crops.parquet` has bounding boxes (and `area_frac`) for whatever subset Step 5 processed (`image_id`, `organ`, `x1/y1/x2/y2`, `found`, `w`, `h`, `area_frac`). Use it locally to crop the already-downloaded images in `data/processed/images/` and:
   - Recompute the **full** Phase-2 classical descriptors (`features/leaf.py`/`bark.py`/`flower.py` - Hu moments + LBP + HSV, not just the quick Hu-moment check from Step 8) on crops and re-run `eval/match_eval.py`.
   - Re-test `imret` (ORB+FAISS) on crops - Phase 3 found it near-chance on whole cluttered images; isolated organ crops remove most of the keypoint noise that caused that.
3. If the comparison in `metadata.json` looks promising: rerun Step 5 with a larger (or `None`) `MAX_PER_GROUP` for full-dataset coverage, then redo Steps 9-16. Cropping could also be combined with the `ce_edge`/SupCon variants from `finetune_colab.ipynb`.
4. If cropping doesn't help (numbers ~unchanged or worse): check Step 7's qualitative crops first. Either the grounding prompts need tuning per organ (revisit Step 4), or - for this dataset - the organ may already fill most of the frame, in which case there's little background left for cropping to remove and the bottleneck is elsewhere.
5. `metadata.json`'s `good_crop_rate_among_detections` (per organ, the fraction of *detected* organs whose box covers at least `min_area_frac` of the frame) is a starting point for a Phase 6 input-validation check: at inference time, run Step 5+6's grounding+area-fraction logic on the user's photo, and if it fails the same `min_area_frac` threshold (or nothing is detected at all), return "not enough <organ> visible in this photo - try getting closer to the <organ>" instead of a low-confidence prediction.
